In [191]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

## ENV SETUP

In [192]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [193]:
anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

In [194]:
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openai = OpenAI()

## MODEL INSTRUCTIONS

In [ ]:
ollama_system_message = """
You are the main assistant for a travel planning application.

Respond to normal user messages and decide which AI model should handle specialized requests.

Use OpenAI when the user needs travel research such as destinations, flights, hotels, restaurants, cuisines, or current prices.

Use Anthropic when the user provides a budget and wants the best travel option within that budget.

Give short and helpful answers.

Do not perform web research or budget optimization yourself.
"""

In [196]:
def analyse_and_get_right_model(destination_city=None, budget=None):
    """
    Analyze the user request and determine which AI model should handle it.
    Returns: 'openai', 'anthropic', or 'llama'
    """
    # If user provides a destination but no budget -> OpenAI for research
    if destination_city is not None and budget is None:
        return "openai"
    # If user provides both destination and budget -> Anthropic for budget optimization
    elif destination_city is not None and budget is not None:
        return "anthropic"
    # If user provides only budget (without destination) -> Anthropic can still help with budget analysis
    elif destination_city is None and budget is not None:
        return "anthropic"
    # Otherwise -> Llama for normal conversation
    else:
        return "llama"

## DATABASE DESIGN

In [ ]:
DB = "travel_plans.db"

# Database creation queries
LOCATION_DB_CREATION_QUERY = """
CREATE TABLE IF NOT EXISTS locations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    city TEXT NOT NULL,
    country TEXT NOT NULL,
    UNIQUE(city, country)
)
"""


# Initialize database
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(LOCATION_DB_CREATION_QUERY)
    conn.commit()
    print("Database initialized successfully")

Database initialized successfully


## TOOL FUNCTIONS

{'success': 'Stored 3 flights, 3 hotels, and 3 restaurants for Paris, France'}


## CHATBOT ARCHITECTURE

The chatbot uses a three-model architecture:

1. **Llama (via Ollama)**: 
   - Handles normal conversation
   - Acts as the dispatcher to determine which specialized model should handle a request
   - Uses tool calling to analyze user intent

2. **OpenAI (GPT-4)**:
   - Specializes in travel research
   - Uses two tool functions:
     - `get_travel_info`: Retrieves data from database
     - `store_travel_info`: Stores research results (max 3 flights, hotels, restaurants per location)
   - Focuses on gathering accurate travel information

3. **Anthropic (Claude)**:
   - Specializes in budget optimization
   - Uses `get_travel_info` to access stored travel data
   - Analyzes all available options and selects the ONE best option within budget
   - Provides detailed cost breakdown (flight, hotel, restaurant, total, remaining budget)

Starting travel planning chatbot...
The chatbot uses:
- Llama (via Ollama) for normal conversation and model selection
- OpenAI for travel research (flights, hotels, restaurants)
- Anthropic for budget optimization and finding the best option

Example queries:
- 'Hello' -> Llama handles normal conversation
- 'Find flights to Paris' -> OpenAI researches travel options
- 'I have $1000 budget for Paris' -> Anthropic finds best option within budget

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/arvindl/Desktop/LLM_Basics/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 861, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arvindl/Desktop/LLM_Basics/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arvindl/Desktop/LLM_Basics/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2695, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arvindl/Desktop/LLM_Basics/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arvindl/Desktop/LLM_Basics/.venv/lib/python3.12/site-packages/gradio/utils.py", l